# 심층 신경망

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # torch import 전에 설정

from collections import OrderedDict

import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

## seed / determinism

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
criterion = nn.CrossEntropyLoss()  # sparse_categorical_crossentropy + softmax 포함

## helpers

In [ ]:
def load_fashion_mnist():
    tr = datasets.FashionMNIST(root="./data", train=True, download=True)
    te = datasets.FashionMNIST(root="./data", train=False, download=True)
    return (tr.data.numpy(), tr.targets.numpy()), (te.data.numpy(), te.targets.numpy())


def to_loader(x, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g)


def keras_init(m):
    # Keras Dense 기본 초기화: glorot_uniform + bias 0
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)


def finalize(model):
    model.apply(keras_init)
    return model.to(device)


def summary(model):
    print(model)
    print("params:", sum(p.numel() for p in model.parameters()))


def fit(model, optimizer, loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
        print(f"Epoch {epoch+1}/{epochs} - loss: {total_loss/n:.4f} - accuracy: {correct/n:.4f}")


def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += criterion(out, yb).item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
    print(f"val loss: {total_loss/n:.4f} - val accuracy: {correct/n:.4f}")
    return total_loss / n, correct / n


# Keras 기본 하이퍼파라미터에 맞춘 옵티마이저 팩토리
def keras_rmsprop(params):  # compile()에 optimizer 미지정 시 기본값
    return torch.optim.RMSprop(params, lr=1e-3, alpha=0.9, eps=1e-7)


def keras_adam(params):
    return torch.optim.Adam(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-7)

## 1) 데이터 (flatten 784) + 은닉층 100 sigmoid 모델

In [ ]:
(train_input, train_target), (test_input, test_target) = load_fashion_mnist()

train_scaled = train_input / 255.0
train_scaled = train_scaled.reshape(-1, 28 * 28)

train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [ ]:
train_loader = to_loader(train_scaled, train_target, shuffle=True)
val_loader = to_loader(val_scaled, val_target, shuffle=False)

In [ ]:
model = finalize(nn.Sequential(
    nn.Linear(784, 100), nn.Sigmoid(),
    nn.Linear(100, 10),
))
summary(model)

이름 지정 버전(name='패션 MNIST 모델'은 PyTorch Sequential에 없으므로 생략)

In [ ]:
model = finalize(nn.Sequential(OrderedDict([
    ("hidden", nn.Linear(784, 100)),
    ("sigmoid", nn.Sigmoid()),
    ("output", nn.Linear(100, 10)),
])))
summary(model)

In [ ]:
optimizer = keras_rmsprop(model.parameters())
fit(model, optimizer, train_loader, epochs=5)

## 2) Flatten + relu 모델 (입력 28x28)

In [ ]:
model = finalize(nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 100), nn.ReLU(),
    nn.Linear(100, 10),
))
summary(model)

In [ ]:
(train_input, train_target), (test_input, test_target) = load_fashion_mnist()
train_scaled = train_input / 255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [ ]:
train_loader = to_loader(train_scaled, train_target, shuffle=True)
val_loader = to_loader(val_scaled, val_target, shuffle=False)

In [ ]:
optimizer = keras_rmsprop(model.parameters())
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)

## 3) 옵티마이저 종류 (Keras 기본값과 동일하게 생성만)

In [ ]:
# sgd = torch.optim.SGD(model.parameters(), lr=0.01)                    # optimizer='sgd' / SGD()
# sgd = torch.optim.SGD(model.parameters(), lr=0.1)                     # learning_rate=0.1
sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

adagrad = torch.optim.Adagrad(model.parameters(), lr=1e-3,
                              initial_accumulator_value=0.1, eps=1e-7)
rmsprop = keras_rmsprop(model.parameters())

## 4) Adam

In [ ]:
model = finalize(nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 100), nn.ReLU(),
    nn.Linear(100, 10),
))
summary(model)

In [ ]:
optimizer = keras_adam(model.parameters())
fit(model, optimizer, train_loader, epochs=5)
evaluate(model, val_loader)